# Pebble v1 — MLM isolation ablation (3 seeds, scaled corpus)

Run the blocks top-to-bottom; state carries across cells. The encoder is adapted ONCE on a big *separate* in-domain corpus (block s3/s4), then BOTH arms are fine-tuned across 3 seeds for a paired delta. Re-run s5–s7 to retune fine-tuning without redoing MLM.

## 0. Install pinned NeoBERT stack  (run once, ~4 min)

In [ ]:
# Cell 1 — pinned NeoBERT stack (Kaggle default torch 2.10 drops sm_60 -> P100 crash).
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
get_ipython().system('pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121')
get_ipython().system('pip install -q transformers==4.48.2 sentencepiece')
print("install cell done")


## 1. Imports & config  (15% masking, 80k corpus cap)

In [ ]:
# Block s1 — imports, seed, config
# Scaled MLM isolation ablation: adapt encoder ONCE on a BIG separate in-domain
# corpus, then fine-tune BOTH arms across 3 seeds -> mean +/- std + paired delta.
import os, random, warnings, urllib.request
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL    = "chandar-lab/NeoBERT"
REVISION = "5424c8efeea6491b151d62dee55a752165407430"
MAX_LEN, BATCH = 64, 32
# --- MLM adaptation budget: BIG separate corpus, standard 15% masking ---
MLM_EPOCHS, MLM_MASK_PROB, MLM_CORPUS_CAP = 2, 0.15, 80000
FT_EPOCHS, FT_PER_POOL = 3, 2500
EMO_VAL_N, SEV_VAL_N = 1000, 600
SEEDS = [13, 42, 1337]
EIREG_NEG = {"anger", "fear", "sadness"}
EIREG_EMOS = ["anger", "fear", "joy", "sadness"]
ART = "/kaggle/working"
set_seed(SEEDS[0])
print(f"config ready | mask={MLM_MASK_PROB} corpus_cap={MLM_CORPUS_CAP} mlm_epochs={MLM_EPOCHS} | device {DEVICE}")


## 2. Downstream data  (GoEmotions -> emotion, EI-reg -> severity) + dedup guard

In [ ]:
# Block s2 — tokenizer + downstream data (GoEmotions simplified -> emotion, EI-reg -> severity)
# Also builds FT_EVAL_TEXTS: the normalized text of every fine-tune/eval example,
# so the MLM corpus (block s3) can be deduped against it (no leakage / no overfit).
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
VOCAB = tok.vocab_size
print("vocab", VOCAB, "| mask_token", tok.mask_token, tok.mask_token_id)

def norm(t):  # canonical form for dedup
    return " ".join(t.lower().split())

def eireg(file_split):
    base = "https://raw.githubusercontent.com/cbaziotis/ntua-slp-semeval2018/master/datasets/task1/EI-reg"
    rows = []
    for emo in EIREG_EMOS:
        url = f"{base}/EI-reg-En-{emo}-{file_split}.txt"
        try:
            raw = urllib.request.urlopen(url, timeout=40).read().decode("utf-8")
        except Exception as e:
            print("eireg download failed:", url, e); return None
        for ln in raw.splitlines()[1:]:
            c = ln.split("\t")
            if len(c) < 4: continue
            try: inten = float(c[3])
            except ValueError: continue
            rows.append({"text": c[1], "severity": inten if emo in EIREG_NEG else 0.0})
    return rows

go_train = load_dataset("go_emotions", "simplified", split="train")
go_val   = load_dataset("go_emotions", "simplified", split="validation")
EMO_NAMES = go_train.features["labels"].feature.names
NEUTRAL = EMO_NAMES.index("neutral"); N_EMO = len(EMO_NAMES)
def go_rows(ds):
    return [{"text": r["text"], "emotion": (r["labels"][0] if r["labels"] else NEUTRAL)} for r in ds]
go_tr, go_va = go_rows(go_train), go_rows(go_val)

ei_tr, ei_va = eireg("train"), eireg("dev")
if not ei_tr or not ei_va:
    print("!! eireg unavailable -> synthetic severity fallback")
    rnd = lambda: random.random()
    ei_tr = [{"text": f"i feel low energy and distress sample {i} {rnd():.2f}", "severity": rnd()} for i in range(4000)]
    ei_va = [{"text": f"a tough day sample {i} {rnd():.2f}", "severity": rnd()} for i in range(600)]
print(f"GoEmotions train={len(go_tr)} val={len(go_va)} ({N_EMO} classes) | EI-reg train={len(ei_tr)} dev={len(ei_va)}")

# text that must NOT appear in the MLM corpus (everything we fine-tune on or eval on)
FT_EVAL_TEXTS = set()
for split in (go_tr, go_va, ei_tr, ei_va):
    FT_EVAL_TEXTS.update(norm(r["text"]) for r in split)
print(f"dedup guard: {len(FT_EVAL_TEXTS)} fine-tune/eval texts blacklisted")

def encode(texts):
    e = tok(texts, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    return e["input_ids"], e["attention_mask"]
print("data ready")


## 3. Build the big SEPARATE MLM corpus  (GoEmotions raw + tweet_eval, deduped)

In [ ]:
# Block s3 — build the BIG, SEPARATE in-domain MLM corpus.
# Sources: GoEmotions *raw* (~211k Reddit comments) + tweet_eval subsets (tweets).
# Every text is deduped against itself AND against FT_EVAL_TEXTS (block s2), so the
# encoder adapts on NEW unlabeled in-domain text -- the real TAPT/DAPT setup.
seen = set(FT_EVAL_TEXTS)   # start from the blacklist; also dedupes within the corpus
corpus, src_counts = [], {}
def add(name, texts):
    kept = 0
    for t in texts:
        if not t: continue
        n = norm(t)
        if n and n not in seen:
            seen.add(n); corpus.append(t); kept += 1
    src_counts[name] = kept
    print(f"  + {name}: +{kept} (corpus={len(corpus)})")

try:
    raw = load_dataset("go_emotions", "raw", split="train")
    add("go_emotions/raw", raw["text"])
except Exception as e:
    print("go_emotions raw failed:", e)

for sub in ["emotion", "sentiment", "offensive", "hate", "irony"]:
    try:
        for sp in ["train", "validation"]:
            ds = load_dataset("tweet_eval", sub, split=sp)
            add(f"tweet_eval/{sub}/{sp}", ds["text"])
    except Exception as e:
        print(f"tweet_eval {sub} failed:", e)

random.shuffle(corpus)
corpus = corpus[:MLM_CORPUS_CAP]
print(f"\n[MLM corpus] {len(corpus)} texts after dedup+cap | sources: {src_counts}")

mlm_ids, mlm_attn = encode(corpus)
print("[MLM corpus] tokenized ->", tuple(mlm_ids.shape))


## 4. MLM pre-training (15% masking) -> save adapted encoder (fp32)
Run once; watch the loss drop.

In [ ]:
# Block s4 — MLM pre-training (15% masking) on the separate corpus, then save the
# adapted encoder in fp32 (no fp16 rounding -> removes the precision confound vs the
# vanilla MLM-off baseline). Run once; block s6 reuses the saved state for every seed.
print(f"[MLM] corpus={len(corpus)} texts, {int(MLM_MASK_PROB*100)}% masking, {MLM_EPOCHS} epochs")
SPECIAL = torch.tensor(tok.all_special_ids)
def mask_batch(ids):
    ids = ids.clone(); labels = ids.clone()
    keep = torch.isin(ids, SPECIAL)
    prob = torch.full(ids.shape, MLM_MASK_PROB); prob[keep] = 0.0
    sel = torch.bernoulli(prob).bool(); labels[~sel] = -100
    r = torch.rand(ids.shape)
    ids[sel & (r < 0.8)] = tok.mask_token_id
    rp = sel & (r >= 0.8) & (r < 0.9); ids[rp] = torch.randint(VOCAB, ids.shape)[rp]
    return ids, labels

# NeoBERTLMHead: .model is the inner encoder; forward -> MaskedLMOutput(logits), no loss.
mlm = AutoModelForMaskedLM.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True).to(DEVICE)
enc_ref = mlm.model
opt = torch.optim.AdamW(mlm.parameters(), lr=5e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(); order = list(range(len(corpus))); mlm.train()
for ep in range(MLM_EPOCHS):
    random.shuffle(order); tot = 0.0; nb = 0
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        ids, labels = mask_batch(mlm_ids[idx])
        ids, labels, attn = ids.to(DEVICE), labels.to(DEVICE), mlm_attn[idx].to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = mlm(input_ids=ids, attention_mask=attn).logits
            loss = F.cross_entropy(logits.view(-1, VOCAB), labels.view(-1), ignore_index=-100)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item(); nb += 1
    print(f"  [MLM] epoch {ep+1}/{MLM_EPOCHS}  loss={tot/nb:.4f}")

adapted_state = {k: v.detach().float().cpu() for k, v in enc_ref.state_dict().items()}
torch.save(adapted_state, f"{ART}/mlm_encoder.pt")
print(f"[MLM] saved adapted encoder -> {ART}/mlm_encoder.pt ({len(adapted_state)} tensors, fp32)")
del mlm, enc_ref, opt; torch.cuda.empty_cache()


## 5. Fine-tune setup — masked two-pool multi-task

In [ ]:
# Block s5 — masked two-pool multi-task fine-tune setup (dataset, model, metrics, finetune fn).
# finetune("MLM-off", None, seed)          -> vanilla NeoBERT
# finetune("MLM-on", adapted_state, seed)  -> encoder adapted in block s4
EMO, SEV = 0, 1
recs  = [{"text": r["text"], "task": EMO, "emo": r["emotion"], "sev": 0.0} for r in go_tr[:FT_PER_POOL]]
recs += [{"text": r["text"], "task": SEV, "emo": -1, "sev": r["severity"]} for r in ei_tr[:FT_PER_POOL]]
class FTDataset(TorchDataset):
    def __init__(self, rs):
        self.ids, self.attn = encode([r["text"] for r in rs])
        self.task = torch.tensor([r["task"] for r in rs])
        self.emo  = torch.tensor([r["emo"] for r in rs], dtype=torch.long)
        self.sev  = torch.tensor([r["sev"] for r in rs], dtype=torch.float)
    def __len__(self): return len(self.task)
    def __getitem__(self, i): return self.ids[i], self.attn[i], self.task[i], self.emo[i], self.sev[i]
ft_ds = FTDataset(recs)
emo_val_ids, emo_val_attn = encode([r["text"] for r in go_va[:EMO_VAL_N]])
emo_val_y = np.array([r["emotion"] for r in go_va[:EMO_VAL_N]])
sev_val_ids, sev_val_attn = encode([r["text"] for r in ei_va[:SEV_VAL_N]])
sev_val_y = np.array([r["severity"] for r in ei_va[:SEV_VAL_N]], dtype=float)

class Head(nn.Module):
    def __init__(s, h, out, d=256, p=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Dropout(p), nn.Linear(h, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, out))
    def forward(s, x): return s.net(x)
class MultiTask(nn.Module):
    def __init__(s, adapted=None):
        super().__init__()
        s.encoder = AutoModel.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
        if adapted is not None: s.encoder.load_state_dict({k: v.float() for k, v in adapted.items()})
        h = getattr(s.encoder.config, "hidden_size", 768)
        s.emotion_head, s.score_head = Head(h, N_EMO), Head(h, 1)
    def forward(s, ids, attn):
        cls = s.encoder(input_ids=ids, attention_mask=attn).last_hidden_state[:, 0, :]
        return s.emotion_head(cls), torch.sigmoid(s.score_head(cls)).squeeze(-1)

def pearson(p, t):  return 0.0 if np.std(p) < 1e-8 or np.std(t) < 1e-8 else float(pearsonr(p, t)[0])
def spearman(p, t): return 0.0 if np.std(p) < 1e-8 or np.std(t) < 1e-8 else float(spearmanr(p, t)[0])
def ece(probs, correct, n=10):
    conf = probs.max(1); b = np.linspace(0, 1, n + 1); e = 0.0; N = len(conf)
    for i in range(n):
        m = (conf > b[i]) & (conf <= b[i + 1])
        if m.sum(): e += m.sum() / N * abs(correct[m].mean() - conf[m].mean())
    return float(e)
@torch.no_grad()
def predict(model, ids, attn):
    model.eval(); oe, os_ = [], []
    for i in range(0, len(ids), BATCH):
        with torch.cuda.amp.autocast():
            el, sp = model(ids[i:i+BATCH].to(DEVICE), attn[i:i+BATCH].to(DEVICE))
        oe.append(torch.softmax(el.float(), -1).cpu().numpy()); os_.append(sp.float().cpu().numpy())
    return np.concatenate(oe), np.concatenate(os_)

METRICS = ["emo_macroF1", "emo_ece", "sev_pearson", "sev_spearman", "sev_mae"]
def finetune(tag, adapted, seed):
    set_seed(seed)
    loader = DataLoader(ft_ds, batch_size=BATCH, shuffle=True)
    model = MultiTask(adapted).to(DEVICE)
    opt = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder.")], "lr": 2e-5},
        {"params": model.encoder.parameters(), "lr": 1e-5}], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    for ep in range(FT_EPOCHS):
        model.train()
        for ids, attn, task, emo, sev in loader:
            ids, attn, task = ids.to(DEVICE), attn.to(DEVICE), task.to(DEVICE)
            emo, sev = emo.to(DEVICE), sev.to(DEVICE)
            me, ms = task == EMO, task == SEV
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                el, sp = model(ids, attn)
                loss = el.sum() * 0.0
                if me.any(): loss = loss + F.cross_entropy(el[me], emo[me])
                if ms.any(): loss = loss + F.mse_loss(sp[ms], sev[ms])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    ep_probs, _ = predict(model, emo_val_ids, emo_val_attn)
    _, sv_pred  = predict(model, sev_val_ids, sev_val_attn)
    pe = ep_probs.argmax(1)
    res = {"arm": tag, "seed": seed,
           "emo_macroF1": float(f1_score(emo_val_y, pe, average="macro")),
           "emo_ece":     ece(ep_probs, (pe == emo_val_y).astype(float)),
           "sev_pearson": pearson(sv_pred, sev_val_y),
           "sev_spearman":spearman(sv_pred, sev_val_y),
           "sev_mae":     float(np.mean(np.abs(sv_pred - sev_val_y)))}
    del model, opt; torch.cuda.empty_cache()
    return res
print("fine-tune setup ready | pools:", len(recs), "| metrics:", METRICS)


## 6. Run both arms across 3 seeds  (incremental CSV checkpoint per seed)

In [ ]:
# Block s6 — run BOTH arms across all seeds. Writes results_per_seed.csv after EVERY
# seed, so if the run is interrupted you keep the seeds already finished.
import pandas as pd
rows = []
csv_seed = f"{ART}/results_per_seed.csv"
for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    off = finetune("MLM-off", None, seed);          print("  off:", {k: round(off[k], 4) for k in METRICS})
    on  = finetune("MLM-on", adapted_state, seed);  print("  on :", {k: round(on[k], 4) for k in METRICS})
    rows += [off, on]
    pd.DataFrame(rows).to_csv(csv_seed, index=False)   # incremental checkpoint
print(f"\nper-seed results saved -> {csv_seed} ({len(rows)} rows)")


## 7. Results table — mean +/- std + paired per-seed delta

In [ ]:
# Block s7 — aggregate: per-arm mean +/- std and the paired per-seed delta (on - off).
df = pd.DataFrame(rows)
summary = []
for arm in ["MLM-off", "MLM-on"]:
    sub = df[df.arm == arm]
    summary.append({"arm": arm, **{m: f"{sub[m].mean():.4f}+/-{sub[m].std():.4f}" for m in METRICS}})
piv = df.pivot(index="seed", columns="arm")
delta = {"arm": "delta (on-off) mean+/-std"}
for m in METRICS:
    d = piv[(m, "MLM-on")] - piv[(m, "MLM-off")]
    delta[m] = f"{d.mean():+.4f}+/-{d.std():.4f}"
sdf = pd.DataFrame(summary + [delta])
sdf.to_csv(f"{ART}/results_summary.csv", index=False)
print("\n================= MLM ABLATION (3 seeds) =================")
print(sdf.to_string(index=False))
print(f"\nartifacts -> {ART}/mlm_encoder.pt , results_per_seed.csv , results_summary.csv")
print("=== RESULT: SUCCESS ===")


## 8. Export a shippable model + inference demo  (train once on chosen arm, save, test)

In [ ]:
# Block s8 — train ONE shippable model on the chosen arm, save it whole, run a demo.
# The ablation (s6) throws every fine-tuned model away; this block keeps one so you
# get a usable artifact (encoder + both heads) plus an analyze(text) inference fn.
SHIP_ARM  = "MLM-off"   # "MLM-off" = vanilla (best severity) | "MLM-on" = adapted (best emotion F1)
SHIP_SEED = 42
adapted = adapted_state if SHIP_ARM == "MLM-on" else None

set_seed(SHIP_SEED)
ship = MultiTask(adapted).to(DEVICE)
opt = torch.optim.AdamW([
    {"params": [p for n, p in ship.named_parameters() if not n.startswith("encoder.")], "lr": 2e-5},
    {"params": ship.encoder.parameters(), "lr": 1e-5}], weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()
loader = DataLoader(ft_ds, batch_size=BATCH, shuffle=True)
for ep in range(FT_EPOCHS):
    ship.train()
    for ids, attn, task, emo, sev in loader:
        ids, attn, task = ids.to(DEVICE), attn.to(DEVICE), task.to(DEVICE)
        emo, sev = emo.to(DEVICE), sev.to(DEVICE)
        me, ms = task == EMO, task == SEV
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            el, sp = ship(ids, attn)
            loss = el.sum() * 0.0
            if me.any(): loss = loss + F.cross_entropy(el[me], emo[me])
            if ms.any(): loss = loss + F.mse_loss(sp[ms], sev[ms])
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    print(f"  [ship/{SHIP_ARM}] epoch {ep+1}/{FT_EPOCHS} done")

# save the WHOLE model + the metadata needed to rebuild it for inference later
ckpt = {"state_dict": {k: v.detach().cpu() for k, v in ship.state_dict().items()},
        "emo_names": EMO_NAMES, "model": MODEL, "revision": REVISION,
        "max_len": MAX_LEN, "arm": SHIP_ARM, "seed": SHIP_SEED}
torch.save(ckpt, f"{ART}/pebble_model.pt")
print(f"[ship] saved -> {ART}/pebble_model.pt  ({SHIP_ARM}, seed {SHIP_SEED}, {len(ckpt['state_dict'])} tensors)")

# ---------------------------------------------------------------- inference
@torch.no_grad()
def analyze(text, topk=3):
    ship.eval()
    ids, attn = encode([text])
    with torch.cuda.amp.autocast():
        el, sp = ship(ids.to(DEVICE), attn.to(DEVICE))
    probs = torch.softmax(el.float(), -1).cpu().numpy()[0]
    sev = float(sp.float().cpu().numpy()[0])
    top = probs.argsort()[::-1][:topk]
    return {"severity": round(sev, 3),
            "emotions": [(EMO_NAMES[i], round(float(probs[i]), 3)) for i in top]}

TESTS = [
    "I just got promoted, best day of my life!",
    "the meeting is at 3pm tomorrow",
    "why does everyone always ignore me",
    "I can't do this anymore, nothing matters",
    "I'm so scared something bad will happen",
]
print("\n================= INFERENCE DEMO =================")
for t in TESTS:
    r = analyze(t)
    top = ", ".join(f"{e}:{p}" for e, p in r["emotions"])
    print(f"severity={r['severity']:.3f} | {top}   <= {t!r}")
print("=== RESULT: SUCCESS ===")
